# 01 Text Processing

Черновик для экспериментов с загрузкой документов, очисткой текста и разбиением на чанки.

In [1]:
def split_by_words(text: str, chunk_size: int, overlap: int) -> list[str]:
    if chunk_size <= overlap:
        raise ValueError("chunk_size должен быть больше overlap")
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        words_chunk = words[i : i + chunk_size]
        chunk_text = ' '.join(words_chunk)
        chunks.append(chunk_text)
        if i + chunk_size >= len(words):
            break
    return chunks

In [2]:
# Создаем тестовый текст ровно из 50 слов (чтобы было удобно считать)
test_text = (
    "Один два три четыре пять шесть семь восемь девять десять "
    "одиннадцать двенадцать тринадцать четырнадцать пятнадцать шестнадцать семнадцать восемнадцать девятнадцать двадцать "
    "двадцать_один двадцать_два двадцать_три двадцать_четыре двадцать_пять двадцать_шесть двадцать_семь двадцать_восемь двадцать_девять тридцать "
    "тридцать_один тридцать_два тридцать_три тридцать_четыре тридцать_пять тридцать_шесть тридцать_семь тридцать_восемь тридцать_девять сорок "
    "сорок_один сорок_два сорок_три сорок_четыре сорок_пять сорок_шесть сорок_семь сорок_восемь сорок_девять пятьдесят"
)

print("--- ТЕСТ 1: chunk_size=20, overlap=0 ---")
chunks_1 = split_by_words(test_text, chunk_size=20, overlap=0)
for i, chunk in enumerate(chunks_1):
    print(f"Чанк {i+1} (длина {len(chunk.split())} слов): {chunk[:40]}...")

print("\n--- ТЕСТ 2: chunk_size=20, overlap=5 ---")
chunks_2 = split_by_words(test_text, chunk_size=20, overlap=5)
for i, chunk in enumerate(chunks_2):
    print(f"Чанк {i+1} (длина {len(chunk.split())} слов): {chunk[:40]}...")

--- ТЕСТ 1: chunk_size=20, overlap=0 ---
Чанк 1 (длина 20 слов): Один два три четыре пять шесть семь восе...
Чанк 2 (длина 20 слов): двадцать_один двадцать_два двадцать_три ...
Чанк 3 (длина 10 слов): сорок_один сорок_два сорок_три сорок_чет...

--- ТЕСТ 2: chunk_size=20, overlap=5 ---
Чанк 1 (длина 20 слов): Один два три четыре пять шесть семь восе...
Чанк 2 (длина 20 слов): шестнадцать семнадцать восемнадцать девя...
Чанк 3 (длина 20 слов): тридцать_один тридцать_два тридцать_три ...


In [3]:
# def split_by_paragraph(text: str, chunk_size: int) -> list[str]:
#     if chunk_size <= 0:
#         raise ValueError("chunk_size должен быть больше 0")
#     paragraphs = text.split('\n\n')
#     chunks = []
#     for i in range(len(paragraphs)):
#         if len(paragraphs[i].split()) <= chunk_size:
#             chunks.append(paragraphs[i])
#         else:
#             f,j = 0,1
#             sentences = paragraphs[i].split('. ')
#             while f == 0:
#                 block = '.'.join(sentences[:j])
#                 if len(block.split()) > chunk_size:
#                     if j == 1: raise ValueError("chunk_size должен быть больше sentence")
#                     block = '.'.join(sentences[:j-1])
#                     chunks.append(block)
#                     sentences = sentences[j:]
#                     j = 0
#                 j += 1
#                 if j == len(sentences):
#                     block = '.'.join(sentences)
#                     chunks.append(block)
#                     f = 1
#     return chunks


In [4]:
def split_by_paragraph(text: str, chunk_size: int) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть больше 0")
    paragraphs = text.split('\n\n')
    chunks = []
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if len(p.split()) <= chunk_size:
            chunks.append(p)
        else:
            sentences = p.split('. ')
            current_chunk_sentences = []
            current_words_count = 0
            for sentence in sentences:
                if not sentence.endswith('.'):
                    sentence += '.'
                sentence_words = len(sentence.split())
                if sentence_words > chunk_size:
                    raise ValueError(f"Предложение слишком длинное ({sentence_words} слов) для chunk_size={chunk_size}")
                if current_words_count + sentence_words <= chunk_size:
                    current_chunk_sentences.append(sentence)
                    current_words_count += sentence_words
                else:
                    if current_chunk_sentences:
                        chunks.append(' '.join(current_chunk_sentences))
                    current_chunk_sentences = [sentence]
                    current_words_count = sentence_words
            if current_chunk_sentences:
                chunks.append(' '.join(current_chunk_sentences))
    return chunks

In [5]:
# 1. Создаем "хитрый" текст для проверки
test_text = """Это первый абзац. Он очень короткий и состоит всего из трех предложений. Скорее всего, он влезет в чанк целиком.

А вот это уже второй абзац, и он специально сделан гораздо длиннее. В нем много слов, чтобы мы могли проверить, как алгоритм разрежет его на отдельные куски, если лимит будет маленьким. Мы должны убедиться, что ни одно слово не потеряется. Алгоритм должен аккуратно сложить предложения в буфер.

Третий абзац — это проверка на прочность. Здесь есть десятичные дроби, например число Pi равно 3.1415, и версия программы 2.0. Если мы использовали неправильный сплит, эти цифры разорвутся. Посмотрим, справится ли наш код с этой задачей."""

# Вспомогательная функция для красивого вывода
def print_chunks(chunks):
    for i, chunk in enumerate(chunks):
        word_count = len(chunk.split())
        print(f"[{i+1}] (слов: {word_count}) -> {chunk}")
    print("-" * 50)

# 2. Запускаем тесты с разными размерами чанков

print("=== ТЕСТ 1: Большой чанк (chunk_size = 50) ===")
# Ожидание: Все абзацы останутся целыми, так как в каждом меньше 50 слов.
chunks_large = split_by_paragraph(test_text, chunk_size=50)
print_chunks(chunks_large)

print("\n=== ТЕСТ 2: Средний чанк (chunk_size = 20) ===")
# Ожидание: Первый абзац останется целым. Второй и третий порежутся на куски по 1-2 предложения.
chunks_medium = split_by_paragraph(test_text, chunk_size=20)
print_chunks(chunks_medium)

print("\n=== ТЕСТ 3: Маленький чанк (chunk_size = 10) ===")
# Ожидание: Жесткая нарезка почти по одному предложению.
chunks_small = split_by_paragraph(test_text, chunk_size=10)
print_chunks(chunks_small)

print("\n=== ТЕСТ 4: Проверка защиты от огромных предложений (chunk_size = 5) ===")
# Ожидание: Код должен упасть с ошибкой ValueError, потому что в тексте есть предложения длиннее 5 слов.
try:
    chunks_error = split_by_paragraph(test_text, chunk_size=5)
    print("Если ты видишь этот текст, значит защита НЕ сработала!")
except ValueError as e:
    print(f"✅ Защита сработала успешно! Ошибка: {e}")

=== ТЕСТ 1: Большой чанк (chunk_size = 50) ===
[1] (слов: 19) -> Это первый абзац. Он очень короткий и состоит всего из трех предложений. Скорее всего, он влезет в чанк целиком.
[2] (слов: 47) -> А вот это уже второй абзац, и он специально сделан гораздо длиннее. В нем много слов, чтобы мы могли проверить, как алгоритм разрежет его на отдельные куски, если лимит будет маленьким. Мы должны убедиться, что ни одно слово не потеряется. Алгоритм должен аккуратно сложить предложения в буфер.
[3] (слов: 36) -> Третий абзац — это проверка на прочность. Здесь есть десятичные дроби, например число Pi равно 3.1415, и версия программы 2.0. Если мы использовали неправильный сплит, эти цифры разорвутся. Посмотрим, справится ли наш код с этой задачей.
--------------------------------------------------

=== ТЕСТ 2: Средний чанк (chunk_size = 20) ===
[1] (слов: 19) -> Это первый абзац. Он очень короткий и состоит всего из трех предложений. Скорее всего, он влезет в чанк целиком.
[2] (слов: 12) -> А вот

ValueError: Предложение слишком длинное (12 слов) для chunk_size=10